# Regular Expressions as Scholarly Method

**Computational Text Analysis Pack · Module 15 — Rule-Based and Symbolic Methods**

*Technique:* Pattern extraction as evidence; precision, recall and the Unicode trap

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/04-rule-based-methods/15a_regex_as_method.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

The course has used regular expressions as plumbing since notebook 00 — the
tokeniser is one. This notebook treats them as what the survey calls them: the
workhorse of corpus construction and pattern extraction, and a scholarly method
in their own right.

The discipline that makes a pattern a method rather than a guess is the same as
for any instrument: state what it is supposed to catch, run it, and then
**measure what it caught and what it missed.**

## The tooling

Python ships `re`. The third-party **`regex`** module adds much better Unicode
support — crucially `\p{L}` style property classes and proper handling of
combining marks, which matters directly for Yorùbá tone marks and Hausa hooked
letters. It is not installed here, so we use `re` with `\w`, which is
Unicode-aware in Python 3 and gets us most of the way.

For pattern work at corpus scale the field's tools are **CQL** (next notebook),
and **GATE/JAPE** for rules that need to see annotations rather than raw
characters.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import re

text = " ".join(df_colonial["text"].head(400))
print(f"working over {len(text):,} characters of colonial-era prose\n")

PATTERNS = {
    "year":        r"\b1[6-9]\d{2}\b",
    "money":       r"[£$]\s?\d[\d,]*",
    "titled_name": r"\b(?:Mr|Mrs|Dr|Sir|Lord|Captain|Chief|King|Colonel)\.?\s+[A-Z]\w+",
    "place_of":    r"\b(?:town|city|village|river|kingdom) of\s+[A-Z]\w+",
    "quotation":   r"\u201c[^\u201d]{10,120}\u201d",
}
for name, pat in PATTERNS.items():
    found = re.findall(pat, text)
    print(f"{name:<14}{len(found):>6} matches   e.g. {found[:3]}")

## Measuring a pattern

A pattern that finds 300 matches has told you nothing until you know how many
are right and how many it missed. Take a slice small enough to check by eye,
and compute precision and recall against it.

In [ ]:
# A hand-checkable slice: the first 20 candidate year-like strings, plus a
# deliberately awkward set of cases the pattern should and should not catch.
probe = ("In 1897 he sailed. The 1900s were hard. Page 1234 of the report. "
         "Between 1801 and 1899, and again in 1066 and in 2011. "
         "He paid \u00a350 in 1885, then \u00a31,200 more.")
year = re.compile(PATTERNS["year"])
gold = {"1897", "1900", "1801", "1899", "1885"}     # 1066 and 2011 are out of range
found = set(year.findall(probe))

tp = found & gold
fp = found - gold
fn = gold - found
precision = len(tp) / len(found) if found else 0
recall = len(tp) / len(gold) if gold else 0
print(f"found     : {sorted(found)}")
print(f"correct   : {sorted(tp)}")
print(f"spurious  : {sorted(fp)}")
print(f"missed    : {sorted(fn)}")
print(f"\nprecision {precision:.2f}   recall {recall:.2f}")
print()
print("'1900s' matched as '1900' — a plural decade counted as a year. That is a")
print("real error class, and you only find it by checking rather than counting.")

## The Unicode trap, again

Notebook 03b showed `[a-z]+` destroying diacritics. The same trap sits inside
every hand-written pattern, and it is worth seeing on a pattern that looks
entirely reasonable.

In [ ]:
sample = ("Ọba Adéyẹmí ti Ọ̀yọ́ àti Ƙungiyar Ɗalibai sun yi taro "
          "with Chief Adeyemi and Mr Okonkwo")

naive = re.findall(r"\b[A-Z][a-z]+\b", sample)
better = re.findall(r"\b\w*[A-ZỌ̀Ẹ][\w\u0300-\u036f]*\b", sample)
print("text            :", sample)
print()
print("naive [A-Z][a-z]+ :", naive)
print("Unicode-aware     :", [w for w in better if w[0].isupper()])
print()
print("The naive capital-letter pattern is the standard first move for finding")
print("proper names in English. On this line it finds the English names and")
print("silently drops the Yoruba and Hausa ones — which, in a corpus of African")
print("text, means your 'named entities' are the colonial ones.")

In [ ]:
# The same failure, measured on the real multilingual corpus.
rows = []
for lang, g in df.groupby("language"):
    joined = " ".join(g["text"].head(60))
    ascii_names = len(re.findall(r"\b[A-Z][a-z]+\b", joined))
    unicode_names = len(re.findall(r"\b[^\W\d_][\w]*\b", joined))
    non_ascii = sum(1 for ch in joined if ord(ch) > 127)
    rows.append({"language": lang, "ascii_capital_matches": ascii_names,
                 "non_ascii_chars": non_ascii})
cover = pd.DataFrame(rows).sort_values("non_ascii_chars", ascending=False)
print(cover.head(10).to_string(index=False))
print()
print("Languages with the most non-ASCII characters are the ones an ASCII")
print("pattern serves worst. That is a sampling bias introduced by a regex.")

### 🔧 Your turn 1

Write a pattern for the phrase 'the X of Y' where both are capitalised, run it on the colonial corpus, and check twenty matches by hand. What is your precision?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Most people land between 0.5 and 0.8, and the errors cluster: sentence-initial
capitals get swept in, and multi-word names ("the Kingdom of Great Britain")
truncate. Both are fixable with more pattern, and each fix costs recall
somewhere else — which is the permanent trade in rule-based extraction. The
scholarly move is not to reach the perfect pattern but to report the pattern,
the sample you checked, and the two numbers.

</details>